Electromechanical Tagging - PV Paiva
=
This notebook re-tags the inverter and string/tracker blocks of the whole PV Paiva plant in the electromechanical drawing. It generates AutoCAD scripts (`.scr`) that re-insert those blocks with the correct attribute values (`UFV`, `PST`, `INV`, `STR`).

**Workflow overview:**
1. Convert the DWG to DXF and load all CAD entities into the DuckDB database (`pv_projects.duckdb`).
2. Query the loaded entities for the project.
3. Number the **inverters** inside each PST (power station), top-to-bottom / left-to-right.
4. Link each **string** to its inverter using the block **color** (in the drawing, strings share the color of the inverter they are connected to, inside the same PST).
5. Number the strings inside each inverter, left-to-right / top-to-bottom.
6. Write two AutoCAD scripts: one to re-insert the trackers and one to re-insert the inverters.

**Tag hierarchy:** `UFV` (plant) → `PST` (power station) → `INV` (inverter) → `STR` (string).

**DWG prerequisites** (the code assumes all of these, and it doesn't check them):
- **File:** the DWG sits in `dwg_folder` and its name (without `.dwg`) matches the `filename` variable in the first code cell (currently `PV Paiva - Eletromecânico - Not tagged`). The ODA File Converter 27.1.0 must be installed at the path set in `DWG_to_DuckDB_ETL.py`.
- **Model space only:** inverters and trackers are block references (`INSERT`) placed directly in model space. Blocks inside other blocks, xrefs or paper space aren't read.
- **Block names:**
  - Every inverter block name contains `inversor` (e.g. `EMF397_inversor`).
  - Every string/tracker block name contains `Tracker` (e.g. `String_Tracker_1x60p_North`).
  - The match is case-sensitive, and no other block may contain these words, or it gets picked up too.
- **Layers:**
  - **Every** inverter and **every** tracker in the plant sits on its PST layer, and the layer name ends with the 2-digit PST number (`EMF_em_pst_09`, not `EMF_em_pst_9`).
  - The PST is read from the last 2 characters of the layer name.
  - A tracker left on any other layer gets a wrong PST and no inverter.
- **Colors:**
  - Inside a PST, each inverter has its own explicit color, and every string connected to it has that same color. The same color can be reused in different PSTs.
  - A color left as ByLayer (256) is shared by all the blocks on the layer, so it can't tell the inverters apart.
  - Two inverters with the same color in the same PST will duplicate their strings in the merge.
- **Attributes:** the script fills the attributes positionally, in the order AutoCAD prompts for them:
  - **Tracker blocks:** `UFV`, `PST`, `INV`, `STR`.
  - **Inverter block:** `UFV`, `PST`, `INV`, followed by four more attributes that are filled with `-`.
- **Scale prompt:**
  - The inverter block has *Scale uniformly* on, so the insert asks for one scale factor.
  - The tracker blocks have it off, so the insert asks for X and Y scale.
  - If this changes, the script values shift into the wrong prompts.
- **When running the `.scr` in AutoCAD:** set `ATTREQ = 1` and `ATTDIA = 0` so the attribute values are read from the script and not from a dialog.

Extracting the AutoCAD data and loading it into the DuckDB database
=
- Defines the folders: `dwg_folder` (where the source DWG is), `temp_folder` (working copy) and `output_folder` (where the DXF is written, the same as `dwg_folder`).
- `filename` holds the drawing name without extension. The DWG, DXF and temp paths are all built from it, so it is the only thing to change when switching drawings.
- Copies the DWG into `temp_folder` so the ODA File Converter only processes that single file and not every DWG in the project folder.
- `convert_dwg_to_dxf` (from `DWG_to_DuckDB_ETL.py`) runs the ODA File Converter to turn the DWG into a DXF (ACAD2018).
- `load_to_duckdb` reads every model space entity from the DXF with `ezdxf` and loads it into the `cad_entities` table under the project name `"PV Paiva"`.
  - If the project already exists, its old rows are deleted first, so the table always holds the latest version of the drawing.
  - Warning: `BOQ.ipynb` and `BOQ - DDB.ipynb` also load their drawings as "PV Paiva". Whichever notebook runs last replaces the other drawing's rows, so re-run this cell before using `data` if one of them ran in between.
  - For block references (`INSERT`) it stores the block name, insertion point (`X`, `Y`), rotation, color and all attribute tags (`UFV`, `PST`, `INV`, `STR`, `GRP`, ...).
- At the end, the temporary DXF and the copied DWG are deleted. The DuckDB connection (`conn`) stays open for the rest of the notebook and is closed in the last cell.

In [1]:
import pandas as pd

from DWG_to_DuckDB_ETL import *
import os
import shutil

dwg_folder = r"C:\Users\Usuario\Desktop\projetos\paiva\teste"
temp_folder = r"C:\Users\Usuario\Desktop\projetos\paiva\temp"
output_folder = dwg_folder

filename = "PV Paiva - Eletromecânico - Not tagged"

os.makedirs(temp_folder, exist_ok=True)
dwg_file = os.path.join(dwg_folder, filename + ".dwg")

shutil.copy(dwg_file, temp_folder)

convert_dwg_to_dxf(temp_folder, output_folder)

dxf_file = os.path.join(output_folder, filename + ".dxf")

temp_dwg_file = os.path.join(temp_folder, filename + ".dwg")

conn = load_to_duckdb(dxf_file, "PV Paiva")
print("Data Loaded")
print("DuckDB Loaded")


os.remove(dxf_file)
os.remove(temp_dwg_file)

Project PV Paiva already exists
Removed existing data for project: PV Paiva
Successfully loaded project
Loaded 12582 entities for project: PV Paiva
Don't forget to close connection.
Data Loaded
DuckDB Loaded


Loading the data from PV Paiva
=
Selects only the columns needed for tagging from `cad_entities` for this project. The result is a DuckDB relation called `data`, which the next cells query directly by its Python variable name (DuckDB can query Python relations/DataFrames by name).

Only `type`, `layer`, `name`, `color`, `rotation`, `X` and `Y` are actually used downstream. `UFV`, `PST`, `INV`, `STR` and `GRP` are all recalculated or overwritten in the following cells.

In [2]:
data = conn.sql("SELECT type, layer, name, color, rotation, X, Y, UFV, PST, INV, STR, GRP FROM cad_entities WHERE project_name = 'PV Paiva'")
print(data)


┌─────────┬───────────────┬────────────────────────────┬─────────┬──────────┬───────────┬───────────┬─────────┬─────────┬─────────┬─────────┬─────────┐
│  type   │     layer     │            name            │  color  │ rotation │     X     │     Y     │   UFV   │   PST   │   INV   │   STR   │   GRP   │
│ varchar │    varchar    │          varchar           │ varchar │  float   │   float   │   float   │ varchar │ varchar │ varchar │ varchar │ varchar │
├─────────┼───────────────┼────────────────────────────┼─────────┼──────────┼───────────┼───────────┼─────────┼─────────┼─────────┼─────────┼─────────┤
│ INSERT  │ EMF_em_pst_22 │ EMF397_PST                 │ 256     │     90.0 │ 34550.234 │ 127242.37 │         │ 22      │ NULL    │ NULL    │ NULL    │
│ INSERT  │ EMF_em_pst_26 │ EMF397_PST                 │ 256     │     90.0 │  34983.15 │ 127069.02 │         │ 23      │ NULL    │ NULL    │ NULL    │
│ INSERT  │ EMF_em_pst_25 │ EMF397_PST                 │ 256     │     90.0 │   35361.0 

Entity count per layer
=
Sanity check: counts the entities on each layer, sorted by count. `show(max_rows=30)` prints every row instead of the truncated default, so all the PST layers (`EMF_em_pst_01` to `EMF_em_pst_29`) are visible.

This helps spot a PST layer that is missing, or one with an unexpected number of blocks, before tagging. A layer here that isn't a PST layer usually means some blocks are on the wrong layer.

In [3]:
result = conn.sql("""
    SELECT
        layer,
        COUNT(*) as count
    FROM data
    GROUP BY layer
    ORDER BY count
""")
result.show(max_rows=30)

┌───────────────┬───────┐
│     layer     │ count │
│    varchar    │ int64 │
├───────────────┼───────┤
│ EMF_em_pst_15 │   417 │
│ EMF_em_pst_27 │   417 │
│ EMF_em_pst_16 │   418 │
│ EMF_em_pst_19 │   419 │
│ EMF_em_pst_26 │   419 │
│ EMF_em_pst_28 │   420 │
│ EMF_em_pst_14 │   420 │
│ EMF_em_pst_17 │   422 │
│ EMF_em_pst_24 │   424 │
│ EMF_em_pst_23 │   424 │
│ EMF_em_pst_22 │   425 │
│ EMF_em_pst_25 │   427 │
│ EMF_em_pst_10 │   427 │
│ EMF_em_pst_29 │   428 │
│ EMF_em_pst_18 │   433 │
│ EMF_em_pst_21 │   434 │
│ EMF_em_pst_20 │   434 │
│ EMF_em_pst_09 │   444 │
│ EMF_em_pst_04 │   446 │
│ EMF_em_pst_11 │   446 │
│ EMF_em_pst_06 │   448 │
│ EMF_em_pst_01 │   448 │
│ EMF_em_pst_03 │   448 │
│ EMF_em_pst_13 │   449 │
│ EMF_em_pst_02 │   449 │
│ EMF_em_pst_05 │   449 │
│ EMF_em_pst_07 │   449 │
│ EMF_em_pst_12 │   449 │
│ EMF_em_pst_08 │   449 │
└───────────────┴───────┘
  29 rows     2 columns



Inverters: selection and preparation
=
- Selects every block whose name contains `inversor` and converts it into a pandas DataFrame.
- `UFV` is fixed to `0` (single plant).
- `PST` is taken from the last two characters of the layer name (`EMF_em_pst_09` → `09`).
- `X_ordering` / `Y_ordering` are the coordinates rounded to 1 decimal. They are used for sorting, so tiny coordinate differences between blocks that are visually aligned don't change the order.
- `color` is cast to `int` (it is stored as text in DuckDB).
- `value_counts()` on `color` shows how many inverters use each color across the plant. Colors are reused between PSTs, so a count above 1 is normal. Inside a single PST, each color must appear only once.

In [4]:
inverters = conn.sql("SELECT * FROM data WHERE name LIKE '%inversor%'")
inverters_df = inverters.fetchdf()
inverters_df['UFV'] = 0
inverters_df['PST'] = inverters_df['layer'].astype(str).str[-2:]
inverters_df['X_ordering'] = round(inverters_df['X'], 1)
inverters_df['Y_ordering'] = round(inverters_df['Y'], 1)
inverters_df['color'] = inverters_df['color'].astype(int)
print(inverters_df)
print(inverters_df['color'].value_counts())

       type          layer             name  color  rotation             X  \
0    INSERT  EMF_em_pst_07  EMF397_inversor     40       0.0  33667.554688   
1    INSERT  EMF_em_pst_07  EMF397_inversor     32       0.0  33727.554688   
2    INSERT  EMF_em_pst_07  EMF397_inversor    244       0.0  33763.554688   
3    INSERT  EMF_em_pst_07  EMF397_inversor    219       0.0  33799.554688   
4    INSERT  EMF_em_pst_07  EMF397_inversor     18       0.0  33871.554688   
..      ...            ...              ...    ...       ...           ...   
459  INSERT  EMF_em_pst_25  EMF397_inversor     18     180.0  35387.085938   
460  INSERT  EMF_em_pst_20  EMF397_inversor      3       0.0  35597.085938   
461  INSERT  EMF_em_pst_26  EMF397_inversor      6       0.0  34859.085938   
462  INSERT  EMF_em_pst_29  EMF397_inversor     40       0.0  34493.085938   
463  INSERT  EMF_em_pst_29  EMF397_inversor      8     180.0  34637.085938   

                 Y  UFV PST INV   STR GRP    X_ordering     Y_o

Inverters: numbering and AutoCAD script
=
- `GRP` = `UFV.PST` (e.g. `0.09`) is the group inside which the inverters are numbered.
- Sort order inside each PST: `Y` descending, then `X` ascending (top-to-bottom, left-to-right).
- `INV` = sequential number inside each `GRP` (`cumcount() + 1`), zero-padded to 2 digits (`01`, `02`, ...).
- `grp.color` = `UFV.PST.color` (e.g. `0.09.16`) is the key that links strings to inverters later on. Each inverter in a PST has its own color, and its strings are drawn in that same color.
- `Script` builds one AutoLISP line per inverter. It re-inserts the block at the same point and rotation, fills `UFV`, `PST` and `INV`, and fills the block's four remaining attributes with `-`:
  `(command-s "._insert" "<block>" "X,Y" 1 <rotation> UFV PST INV "-" "-" "-" "-")`
  (`1` is the scale. The tracker script passes `1 1`, X and Y scale, because of how the tracker blocks are defined.)

In [5]:
inverters_df['GRP'] = inverters_df["UFV"].astype(str) + "." + inverters_df["PST"].astype(str)
inverters_df.sort_values(by=['GRP', 'Y_ordering', 'X_ordering'], ascending=[True, False, True], inplace=True, ignore_index=True)
inverters_df["INV"] = (inverters_df.groupby("GRP").cumcount()+1).astype(str).str.zfill(2)
inverters_df['grp.color'] = inverters_df['GRP'].astype(str) + "." + inverters_df['color'].astype(str)

inverters_df['Script'] = inverters_df.apply(
    lambda row: (
        f'(command-s "._insert" "{row['name']}" '
        f'"{row["X"]},{row["Y"]}" 1 {row['rotation']} {str(row["UFV"]).zfill(2)} {str(row["PST"]).zfill(2)} {str(row["INV"]).zfill(2)} "-" "-" "-" "-")'
    ),
    axis=1
)

print(inverters_df[inverters_df['grp.color'].duplicated(keep=False)])
print(inverters_df)



Empty DataFrame
Columns: [type, layer, name, color, rotation, X, Y, UFV, PST, INV, STR, GRP, X_ordering, Y_ordering, grp.color, Script]
Index: []
       type          layer             name  color  rotation             X  \
0    INSERT  EMF_em_pst_01  EMF397_inversor      7     180.0  36005.207031   
1    INSERT  EMF_em_pst_01  EMF397_inversor      8     180.0  36065.207031   
2    INSERT  EMF_em_pst_01  EMF397_inversor      6     180.0  35813.207031   
3    INSERT  EMF_em_pst_01  EMF397_inversor      4     180.0  35681.503906   
4    INSERT  EMF_em_pst_01  EMF397_inversor      5     180.0  35735.207031   
..      ...            ...              ...    ...       ...           ...   
459  INSERT  EMF_em_pst_29  EMF397_inversor    116     180.0  34271.085938   
460  INSERT  EMF_em_pst_29  EMF397_inversor    177     180.0  34349.085938   
461  INSERT  EMF_em_pst_29  EMF397_inversor    219     180.0  34433.085938   
462  INSERT  EMF_em_pst_29  EMF397_inversor    244     180.0  34511.085938

Strings/trackers: selection and preparation
=
Selects every block whose name contains `Tracker` (all string/tracker blocks in the plant). It applies the same preparation as the inverters: `UFV = 0`, and `PST` from the last two characters of the layer.

Note: there is no layer filter, so every tracker must be on its `EMF_em_pst_XX` layer. A tracker on any other layer gets a meaningless PST (e.g. `EMF_em_tracker_64p` → `4p`), and it won't find an inverter in the merge.

In [6]:
strings = conn.sql("SELECT * FROM data WHERE name LIKE '%Tracker%'")
strings_df = strings.fetchdf()
strings_df['UFV'] = 0
strings_df['PST'] = strings_df['layer'].astype(str).str[-2:]
print(strings_df)

         type          layer                        name color  rotation  \
0      INSERT  EMF_em_pst_07  String_Tracker_1x60p_North   219       0.0   
1      INSERT  EMF_em_pst_07  String_Tracker_1x60p_North   219     180.0   
2      INSERT  EMF_em_pst_07  String_Tracker_1x60p_North   219       0.0   
3      INSERT  EMF_em_pst_07  String_Tracker_1x60p_North   219     180.0   
4      INSERT  EMF_em_pst_07  String_Tracker_1x60p_North   219       0.0   
...       ...            ...                         ...   ...       ...   
12084  INSERT  EMF_em_pst_14  String_Tracker_1x30p_South     4       0.0   
12085  INSERT  EMF_em_pst_14  String_Tracker_1x60p_South     2       0.0   
12086  INSERT  EMF_em_pst_14  String_Tracker_1x60p_South     2     180.0   
12087  INSERT  EMF_em_pst_14  String_Tracker_1x30p_South     2       0.0   
12088  INSERT  EMF_em_pst_14  String_Tracker_1x30p_South     2       0.0   

                  X              Y  UFV PST INV STR   GRP  
0      33823.554688  128974

Creating the link key
=
- `grp.color` = `UFV.PST.color` builds, for every string in the plant, the same key used for the inverters (e.g. `0.07.219`).
- Drops the old `INV` column (read from the DWG) so the merge in the next cell doesn't create duplicate `INV_x` / `INV_y` columns.

In [7]:
strings_df['grp.color'] = strings_df['UFV'].astype(str) + "." + strings_df['PST'].astype(str) + '.' + strings_df['color'].astype(str)
strings_df.drop(columns=['INV'], inplace=True)
print(strings_df)

         type          layer                        name color  rotation  \
0      INSERT  EMF_em_pst_07  String_Tracker_1x60p_North   219       0.0   
1      INSERT  EMF_em_pst_07  String_Tracker_1x60p_North   219     180.0   
2      INSERT  EMF_em_pst_07  String_Tracker_1x60p_North   219       0.0   
3      INSERT  EMF_em_pst_07  String_Tracker_1x60p_North   219     180.0   
4      INSERT  EMF_em_pst_07  String_Tracker_1x60p_North   219       0.0   
...       ...            ...                         ...   ...       ...   
12084  INSERT  EMF_em_pst_14  String_Tracker_1x30p_South     4       0.0   
12085  INSERT  EMF_em_pst_14  String_Tracker_1x60p_South     2       0.0   
12086  INSERT  EMF_em_pst_14  String_Tracker_1x60p_South     2     180.0   
12087  INSERT  EMF_em_pst_14  String_Tracker_1x30p_South     2       0.0   
12088  INSERT  EMF_em_pst_14  String_Tracker_1x30p_South     2       0.0   

                  X              Y  UFV PST STR   GRP grp.color  
0      33823.554688  

Linking each string to its inverter
=
Left-joins all the strings with the inverters on `grp.color`, so each string gets the `INV` number of the inverter that has the same color in the same PST. The result is `emf`, which holds every string in the plant.

- Strings with no match (a color with no inverter in that PST, or a tracker on the wrong layer) end up with `INV = NaN`. Check for these before generating the script, e.g. `emf[emf['INV'].isna()]`.
- If two inverters in the same PST share a color, the merge duplicates the matching strings. In that case `emf` has more rows than `strings_df`.
- The commented-out `to_excel` line exports `emf` to the Desktop for manual checking when needed.

In [8]:
inverters_subset = inverters_df[['grp.color', 'INV']]
emf = strings_df.merge(inverters_subset, on='grp.color', how='left')
print(emf.columns)
print(emf)
#print(emf[emf['INV'].isna()])
#emf.to_excel(r"C:\Users\Usuario\Desktop\emf.xlsx")

Index(['type', 'layer', 'name', 'color', 'rotation', 'X', 'Y', 'UFV', 'PST',
       'STR', 'GRP', 'grp.color', 'INV'],
      dtype='str')
         type          layer                        name color  rotation  \
0      INSERT  EMF_em_pst_07  String_Tracker_1x60p_North   219       0.0   
1      INSERT  EMF_em_pst_07  String_Tracker_1x60p_North   219     180.0   
2      INSERT  EMF_em_pst_07  String_Tracker_1x60p_North   219       0.0   
3      INSERT  EMF_em_pst_07  String_Tracker_1x60p_North   219     180.0   
4      INSERT  EMF_em_pst_07  String_Tracker_1x60p_North   219       0.0   
...       ...            ...                         ...   ...       ...   
12084  INSERT  EMF_em_pst_14  String_Tracker_1x30p_South     4       0.0   
12085  INSERT  EMF_em_pst_14  String_Tracker_1x60p_South     2       0.0   
12086  INSERT  EMF_em_pst_14  String_Tracker_1x60p_South     2     180.0   
12087  INSERT  EMF_em_pst_14  String_Tracker_1x30p_South     2       0.0   
12088  INSERT  EMF_em_pst_

Strings: numbering and AutoCAD script
=
- `GRP` = `UFV.PST.INV` (e.g. `0.01.01`) is the group inside which the strings are numbered, so string numbering restarts for each inverter.
- Sort order inside each inverter: `X` ascending, then `Y` descending (left-to-right, top-to-bottom). The rounded `X_ordering` / `Y_ordering` coordinates are used, as for the inverters.
- `STR` = sequential number inside each `GRP`, zero-padded to 2 digits.
- `Script` builds the tracker re-insert line with the same structure as the inverter script, but with `1 1` (X and Y scale) before the rotation, and the `STR` value in place of the `-` placeholders:
  `(command-s "._insert" "<block>" "X,Y" 1 1 <rotation> UFV PST INV STR)`

In [9]:
emf['GRP'] = emf["UFV"].astype(str) + "." + emf["PST"].astype(str) + "." + emf["INV"].astype(str)
emf['X_ordering'] = round(emf['X'], 1)
emf['Y_ordering'] = round(emf['Y'], 1)
emf.sort_values(by=['GRP', 'X_ordering', 'Y_ordering'], ascending=[True, True, False], inplace=True, ignore_index=True)
emf["STR"] = (emf.groupby("GRP").cumcount()+1).astype(str).str.zfill(2)


emf['Script'] = emf.apply(
    lambda row: (
        f'(command-s "._insert" "{row['name']}" '
        f'"{row["X"]},{row["Y"]}" 0.001 0.001 {row["rotation"]} {str(row["UFV"]).zfill(2)} {str(row["PST"]).zfill(2)} {str(row["INV"]).zfill(2)} {str(row["STR"]).zfill(2)})'
    ),
    axis=1
)
print(emf)

         type          layer                        name color  rotation  \
0      INSERT  EMF_em_pst_01  String_Tracker_1x30p_North     7       0.0   
1      INSERT  EMF_em_pst_01  String_Tracker_1x30p_North     7       0.0   
2      INSERT  EMF_em_pst_01  String_Tracker_1x30p_North     7       0.0   
3      INSERT  EMF_em_pst_01  String_Tracker_1x30p_North     7       0.0   
4      INSERT  EMF_em_pst_01  String_Tracker_1x30p_North     7       0.0   
...       ...            ...                         ...   ...       ...   
12084  INSERT  EMF_em_pst_29  String_Tracker_1x60p_South     9     180.0   
12085  INSERT  EMF_em_pst_29  String_Tracker_1x30p_South     9       0.0   
12086  INSERT  EMF_em_pst_29  String_Tracker_1x30p_South     9       0.0   
12087  INSERT  EMF_em_pst_29  String_Tracker_1x30p_South     9       0.0   
12088  INSERT  EMF_em_pst_29  String_Tracker_1x30p_South     9       0.0   

                  X              Y  UFV PST STR      GRP grp.color INV  \
0      35957.

Writing the AutoCAD scripts
=
Writes two `.scr` files for the whole plant:
- **Reinserting Trackers - Paiva.scr** from `emf` (all strings)
- **Reinserting Inverters - Paiva.scr** from `inverters_df` (all inverters)

For each layer, the script first sets that layer as current (`._layer set`) and then writes all the insert lines for the blocks on it, so the re-inserted blocks land on their original PST layers.

To use the scripts in AutoCAD, delete the original blocks first (otherwise they end up duplicated) and then run the `.scr` files with the `SCRIPT` command.

Finally, the DuckDB connection is closed.

In [10]:
with open(r"C:\Users\Usuario\Desktop\Pessoal\Data Engineering\Python\Ecotechne\Tagueamento Inversor e Estruturas\Reinserting Trackers - Paiva.scr", 'w') as f:
    for i in emf['layer'].unique():
        mask = emf['layer'] == i

        if mask.any():
            f.write(f'(command-s "._layer" "set" "{i}" "")\n')
            f.write('\n'.join(emf[mask]['Script'].astype(str)) + '\n')

with open(r"C:\Users\Usuario\Desktop\Pessoal\Data Engineering\Python\Ecotechne\Tagueamento Inversor e Estruturas\Reinserting Inverters - Paiva.scr", 'w') as f:
    for i in inverters_df['layer'].unique():
        mask = inverters_df['layer'] == i

        if mask.any():
            f.write(f'(command-s "._layer" "set" "{i}" "")\n')
            f.write('\n'.join(inverters_df[mask]['Script'].astype(str)) + '\n')


conn.close()